<a href="https://colab.research.google.com/github/Mehak1301/customer-retention-churn-analysis/blob/main/customer_retention_churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df= pd.read_csv('churn_flagged.csv')

In [3]:
"""
CHURN PREDICTION MODEL"""


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)


In [7]:

# LOAD DATA
df = pd.read_csv("churn_flagged.csv", header=None,
                  names=["customer_id", "recency_days", "frequency", "monetary", "churned"])

print("Rows loaded:", len(df))
print(df.head())
print("\nChurn rate in dataset:", round(df['churned'].mean() * 100, 1), "%")


Rows loaded: 5878
   customer_id  recency_days  frequency  monetary  churned
0        15984             2         30   8759.32        0
1        14656           169          5    814.65        1
2        16316            64         13   5744.91        0
3        18237             2          5    987.10        0
4        15652            91          5   1420.34        1

Churn rate in dataset: 50.9 %


In [14]:
# FEATURES AND TARGET
# recency_days, frequency, monetary are RFM features.

features = ["frequency", "monetary"]
X = df[features]
y = df["churned"]


In [15]:

# TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [16]:
# TRAIN MODELS
# --- Logistic Regression ---
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)
log_reg_preds = log_reg.predict(X_test_scaled)
log_reg_probs = log_reg.predict_proba(X_test_scaled)[:, 1]

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)   # tree models don't need scaling
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]


In [17]:

# EVALUATE BOTH MODELS

def evaluate(name, y_true, preds, probs):
    print(f"\n===== {name} =====")
    print("Accuracy :", round(accuracy_score(y_true, preds), 3))
    print("Precision:", round(precision_score(y_true, preds), 3))
    print("Recall   :", round(recall_score(y_true, preds), 3))
    print("F1 Score :", round(f1_score(y_true, preds), 3))
    print("AUC      :", round(roc_auc_score(y_true, probs), 3))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, preds))
    print("\nClassification Report:")
    print(classification_report(y_true, preds))

evaluate("Logistic Regression", y_test, log_reg_preds, log_reg_probs)
evaluate("Random Forest", y_test, rf_preds, rf_probs)




===== Logistic Regression =====
Accuracy : 0.687
Precision: 0.654
Recall   : 0.816
F1 Score : 0.726
AUC      : 0.769

Confusion Matrix:
[[320 258]
 [110 488]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.55      0.63       578
           1       0.65      0.82      0.73       598

    accuracy                           0.69      1176
   macro avg       0.70      0.68      0.68      1176
weighted avg       0.70      0.69      0.68      1176


===== Random Forest =====
Accuracy : 0.62
Precision: 0.629
Recall   : 0.617
F1 Score : 0.623
AUC      : 0.681

Confusion Matrix:
[[360 218]
 [229 369]]

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.62      0.62       578
           1       0.63      0.62      0.62       598

    accuracy                           0.62      1176
   macro avg       0.62      0.62      0.62      1176
weighted avg       0.62      0.62      0

In [19]:
# FEATURE IMPORTANCE (Random Forest)
importance_df = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("\n===== Feature Importance (Random Forest) =====")
print(importance_df)



===== Feature Importance (Random Forest) =====
     feature  importance
1   monetary    0.843415
0  frequency    0.156585


In [20]:

# SAVE RESULTS FOR YOUR RESUME / README
# Whichever model performs better (usually compare AUC), use its numbers.
#   - "Random Forest churn model achieved 87% accuracy / 0.84 AUC"
#   - "Recency was the strongest predictor of churn, followed by frequency"

results_summary = pd.DataFrame({
    "model": ["Logistic Regression", "Random Forest"],
    "accuracy": [accuracy_score(y_test, log_reg_preds), accuracy_score(y_test, rf_preds)],
    "auc": [roc_auc_score(y_test, log_reg_probs), roc_auc_score(y_test, rf_probs)],
})
results_summary.to_csv("model_results_summary.csv", index=False)
importance_df.to_csv("feature_importance.csv", index=False)

print("\nSaved model_results_summary.csv and feature_importance.csv")



Saved model_results_summary.csv and feature_importance.csv
